In [3]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
from sklearn.metrics import confusion_matrix

# Disable eager execution for TensorFlow 1.x compatibility
tf.compat.v1.disable_eager_execution()

# Load MNIST dataset
mnist, info = tfds.load('mnist', with_info=True, as_supervised=True)

def preprocess(image, label, n_input=784, n_classes=10):
    image = tf.reshape(image, [n_input])
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, n_classes)
    return image, label

# Define Hyperparameter variations


In [6]:
# ****************************************************************************************************************************

# Define hyperparameter variations
# activation_functions = {'relu': tf.nn.relu, 'sigmoid': tf.nn.sigmoid, 'tanh': tf.nn.tanh} # Uncomment for all activation functions
activation_functions = {'relu': tf.nn.relu} # using only ReLU activation function

# hidden_layer_sizes_single_layer = [256, 128, 64] # Uncomment for single layer
# hidden_layer_sizes_single_layer = [256]

hidden_layer_sizes_double_layer = [(160,100), (100,160), (100,100), (100,60), (60,60)] # Uncomment for double layer
# hidden_layer_sizes_double_layer = [(160,100)] # using only (160,100)

learning_rates = [1, 0.1 , 0.01, 0.001] # Uncomment for all learning rates
# learning_rates = [1] # Using only 0.1 learning rate

# batch_sizes = [100, 10, 1] # Uncomment for all batch sizes
batch_sizes = [10] # Using only 10 batch size

# epochs_list = [100, 50, 10] # Uncomment for all epochs
epochs_list = [50] # Using only 50 epochs

# ****************************************************************************************************************************

In [7]:
print(
    '''
    Hyperparameter variations:
    Activation functions: {}
    Hidden layer sizes: {}
    Learning rates: {}
    Batch sizes: {}
    Epochs: {}
    '''.format(activation_functions.keys(), hidden_layer_sizes_double_layer, learning_rates, batch_sizes, epochs_list)
)


    Hyperparameter variations:
    Activation functions: dict_keys(['relu'])
    Hidden layer sizes: [(160, 100), (100, 160), (100, 100), (100, 60), (60, 60)]
    Learning rates: [1, 0.1, 0.01, 0.001]
    Batch sizes: [10]
    Epochs: [50]
    


In [8]:
# Create results directory
os.makedirs("results", exist_ok=True)

In [9]:
# Save metrics in txt
with open(f"results/HyperparameterVariations.txt", "w") as f:
    f.write('''Hyperparameter variations:
Activation functions: {}
Hidden layer sizes: {}
Learning rates: {}
Batch sizes: {}
Epochs: {}
'''.format(activation_functions.keys(), hidden_layer_sizes_double_layer, learning_rates, batch_sizes, epochs_list)
)

# This plots curves in epochs


In [ ]:
# Loop through all combinations
for act_name, activation in activation_functions.items():
    for hidden_size in hidden_layer_sizes_double_layer:
        for lr in learning_rates:
            for batch_size in batch_sizes:
                for epochs in epochs_list:
                    
                    # Prepare dataset
                    train_data = mnist['train'].map(preprocess).shuffle(60000).batch(batch_size)
                    test_data = mnist['test'].map(preprocess).batch(batch_size)
                    
                    # Define placeholders
                    X = tf.compat.v1.placeholder(tf.float32, [None, 784])
                    Y = tf.compat.v1.placeholder(tf.float32, [None, 10])
                    
                    # Define weights and biases
                    weights = {
                        'h1': tf.Variable(tf.random.normal([784, hidden_size[0]])),
                        'h2': tf.Variable(tf.random.normal([hidden_size[0], hidden_size[1]])),
                        'out': tf.Variable(tf.random.normal([hidden_size[1], 10]))
                    }
                    biases = {
                        'b1': tf.Variable(tf.random.normal([hidden_size[0]])),
                        'b2': tf.Variable(tf.random.normal([hidden_size[1]])),
                        'out': tf.Variable(tf.random.normal([10]))
                    }
                    
                    # Define neural network
                    def neural_net(x):
                        layer_1 = activation(tf.add(tf.matmul(x, weights['h1']), biases['b1']))
                        layer_2 = activation(tf.add(tf.matmul(layer_1, weights['h2']), biases['b2']))
                        out_layer = tf.matmul(layer_2, weights['out']) + biases['out']
                        return out_layer
                    
                    logits = neural_net(X)
                    loss_op = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=Y))
                    optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate=lr).minimize(loss_op)
                    correct_pred = tf.equal(tf.argmax(logits, 1), tf.argmax(Y, 1))
                    accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))
                    
                    init = tf.compat.v1.global_variables_initializer()
                    
                    # Lists for storing performance data
                    loss_history, accuracy_history, y_true, y_pred = [], [], [], []
                    start_time = time.time()
                    
                    with tf.compat.v1.Session() as sess:
                        sess.run(init)
                        train_iterator = tf.compat.v1.data.make_initializable_iterator(train_data)
                        next_train_element = train_iterator.get_next()
                        sess.run(train_iterator.initializer)
                        
                        for epoch in range(epochs):
                            epoch_loss, epoch_acc, batch_count = 0, 0, 0
                            while True:
                                try:
                                    batch_x, batch_y = sess.run(next_train_element)
                                    _, loss, acc = sess.run([optimizer, loss_op, accuracy], feed_dict={X: batch_x, Y: batch_y})
                                    epoch_loss += loss
                                    epoch_acc += acc
                                    batch_count += 1
                                except tf.errors.OutOfRangeError:
                                    break
                            epoch_loss /= batch_count
                            epoch_acc /= batch_count
                            loss_history.append(epoch_loss)
                            accuracy_history.append(epoch_acc)
                            sess.run(train_iterator.initializer)
                            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.3f}")
                        
                        # Evaluate on test data
                        test_iterator = tf.compat.v1.data.make_initializable_iterator(test_data)
                        next_test_element = test_iterator.get_next()
                        sess.run(test_iterator.initializer)
                        test_acc, test_count = 0, 0
                        
                        while True:
                            try:
                                test_x, test_y = sess.run(next_test_element)
                                acc, preds = sess.run([accuracy, tf.argmax(logits, 1)], feed_dict={X: test_x, Y: test_y})
                                y_true.extend(np.argmax(test_y, axis=1))
                                y_pred.extend(preds)
                                test_acc += acc
                                test_count += 1
                            except tf.errors.OutOfRangeError:
                                break
                        test_acc /= test_count
                        execution_time = time.time() - start_time
                        
                        # Save results
                        folder_name = f"results/{act_name}_H{hidden_size[0]}_{hidden_size[1]}_LR{lr}_B{batch_size}_E{epochs}"
                        os.makedirs(folder_name, exist_ok=True)
                        
                        # Plot loss curve
                        plt.figure()
                        plt.plot(range(epochs), loss_history, label='Loss')
                        plt.xlabel('Epochs')
                        plt.ylabel('Loss')
                        plt.title('Loss Curve')
                        plt.legend()
                        plt.savefig(f"{folder_name}/loss_curve.png")
                        plt.close()
                        
                        # Plot accuracy curve
                        plt.figure()
                        plt.plot(range(epochs), accuracy_history, label='Accuracy')
                        plt.xlabel('Epochs')
                        plt.ylabel('Accuracy')
                        plt.title('Accuracy Curve')
                        plt.legend()
                        plt.savefig(f"{folder_name}/accuracy_curve.png")
                        plt.close()
                        
                        # Plot and save confusion matrix
                        cm = confusion_matrix(y_true, y_pred)
                        plt.figure(figsize=(8,6))
                        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
                        plt.xlabel('Predicted')
                        plt.ylabel('Actual')
                        plt.title('Confusion Matrix')
                        plt.savefig(f"{folder_name}/confusion_matrix.png")
                        plt.close()
                        
                        # Save metrics in txt
                        with open(f"{folder_name}/metrics.txt", "w") as f:
                            f.write(f"ActivationFunction,HiddenSize,LearningRate,BatchSize,NunberOfEpochs,Test Accuracy,Execution Time(in sec)/n{act_name},({hidden_size[0]} and {hidden_size[1]}),{lr},{batch_size},{epochs},{test_acc},{execution_time}")
                        
                        # Save metrics in csv
                        with open(f"{folder_name}/metrics.csv", "w") as f:
                            f.write(f"ActivationFunction,HiddenSize,LearningRate,BatchSize,NunberOfEpochs,Test Accuracy,Execution Time(in sec)/n{act_name},({hidden_size[0]} and {hidden_size[1]}),{lr},{batch_size},{epochs},{test_acc},{execution_time}")
                            

Epoch 1/50 - Loss: 35.6899, Accuracy: 0.810
Epoch 2/50 - Loss: 7.0066, Accuracy: 0.912
Epoch 3/50 - Loss: 3.4699, Accuracy: 0.936
Epoch 4/50 - Loss: 2.1602, Accuracy: 0.949
Epoch 5/50 - Loss: 1.4346, Accuracy: 0.959
Epoch 6/50 - Loss: 0.9678, Accuracy: 0.965
Epoch 7/50 - Loss: 0.7378, Accuracy: 0.971
Epoch 8/50 - Loss: 0.5554, Accuracy: 0.975
Epoch 9/50 - Loss: 0.4776, Accuracy: 0.977
Epoch 10/50 - Loss: 0.3697, Accuracy: 0.980
Epoch 11/50 - Loss: 0.3052, Accuracy: 0.983
Epoch 12/50 - Loss: 0.2891, Accuracy: 0.983
Epoch 13/50 - Loss: 0.2512, Accuracy: 0.984
Epoch 14/50 - Loss: 0.2108, Accuracy: 0.986
Epoch 15/50 - Loss: 0.1967, Accuracy: 0.987
Epoch 16/50 - Loss: 0.1728, Accuracy: 0.989
Epoch 17/50 - Loss: 0.1630, Accuracy: 0.989
Epoch 18/50 - Loss: 0.1503, Accuracy: 0.990
Epoch 19/50 - Loss: 0.1456, Accuracy: 0.990
Epoch 20/50 - Loss: 0.1440, Accuracy: 0.991
Epoch 21/50 - Loss: 0.1317, Accuracy: 0.991
Epoch 22/50 - Loss: 0.1274, Accuracy: 0.992
Epoch 23/50 - Loss: 0.1085, Accuracy: 0.

In [ ]:
'''
Hyperparameter variations:
Activation functions: dict_keys(['relu'])
Hidden layer sizes: [(160, 100), (100, 160), (100, 100), (100, 60), (60, 60)]
Learning rates: [0.1,0.01,0.001]
Batch sizes: [10]
Epochs: [50]
'''

# Loop through all combinations
for act_name, activation in activation_functions.items():
    for hidden_size in hidden_layer_sizes_double_layer:
        for lr in learning_rates:
            for batch_size in batch_sizes:
                for epochs in epochs_list:
                    
                    # Prepare dataset
                    train_data = mnist['train'].map(preprocess).shuffle(60000).batch(batch_size)
                    test_data = mnist['test'].map(preprocess).batch(batch_size)
                    
                    # Define placeholders
                    X = tf.compat.v1.placeholder(tf.float32, [None, 784])
                    Y = tf.compat.v1.placeholder(tf.float32, [None, 10])
                    
                    # Define weights and biases
                    weights = {
                        'h1': tf.Variable(tf.random.normal([784, hidden_size[0]])),
                        'h2': tf.Variable(tf.random.normal([hidden_size[0], hidden_size[1]])),
                        'out': tf.Variable(tf.random.normal([hidden_size[1], 10]))
                    }
                    biases = {
                        'b1': tf.Variable(tf.random.normal([hidden_size[0]])),
                        'b2': tf.Variable(tf.random.normal([hidden_size[1]])),
                        'out': tf.Variable(tf.random.normal([10]))
                    }
                    
                    # Define neural network
                    def neural_net(x):
                        layer_1 = activation(tf.add(tf.matmul(x, weights['h1']), biases['b1']))
                        layer_2 = activation(tf.add(tf.matmul(layer_1, weights['h2']), biases['b2']))
                        out_layer = tf.matmul(layer_2, weights['out']) + biases['out']
                        return out_layer
                    
                    logits = neural_net(X)
                    loss_op = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=Y))
                    optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate=lr).minimize(loss_op)
                    correct_pred = tf.equal(tf.argmax(logits, 1), tf.argmax(Y, 1))
                    accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))
                    
                    init = tf.compat.v1.global_variables_initializer()
                    
                    # Lists for storing performance data
                    loss_history, accuracy_history, y_true, y_pred = [], [], [], []
                    start_time = time.time()
                    
                    with tf.compat.v1.Session() as sess:
                        sess.run(init)
                        train_iterator = tf.compat.v1.data.make_initializable_iterator(train_data)
                        next_train_element = train_iterator.get_next()
                        sess.run(train_iterator.initializer)
                        
                        for epoch in range(epochs):
                            epoch_loss, epoch_acc, batch_count = 0, 0, 0
                            while True:
                                try:
                                    batch_x, batch_y = sess.run(next_train_element)
                                    _, loss, acc = sess.run([optimizer, loss_op, accuracy], feed_dict={X: batch_x, Y: batch_y})
                                    epoch_loss += loss
                                    epoch_acc += acc
                                    batch_count += 1
                                except tf.errors.OutOfRangeError:
                                    break
                            epoch_loss /= batch_count
                            epoch_acc /= batch_count
                            loss_history.append(epoch_loss)
                            accuracy_history.append(epoch_acc)
                            sess.run(train_iterator.initializer)
                            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.3f}")
                        
                        # Evaluate on test data
                        test_iterator = tf.compat.v1.data.make_initializable_iterator(test_data)
                        next_test_element = test_iterator.get_next()
                        sess.run(test_iterator.initializer)
                        test_acc, test_count = 0, 0
                        
                        while True:
                            try:
                                test_x, test_y = sess.run(next_test_element)
                                acc, preds = sess.run([accuracy, tf.argmax(logits, 1)], feed_dict={X: test_x, Y: test_y})
                                y_true.extend(np.argmax(test_y, axis=1))
                                y_pred.extend(preds)
                                test_acc += acc
                                test_count += 1
                            except tf.errors.OutOfRangeError:
                                break
                        test_acc /= test_count
                        execution_time = time.time() - start_time
                        
                        # Save results
                        folder_name = f"results/{act_name}_H{hidden_size[0]}_{hidden_size[1]}_LR{lr}_B{batch_size}_E{epochs}"
                        os.makedirs(folder_name, exist_ok=True)
                        
                        # Plot loss curve
                        plt.figure()
                        plt.plot(range(epochs), loss_history, label='Loss')
                        plt.xlabel('Epochs')
                        plt.ylabel('Loss')
                        plt.title('Loss Curve')
                        plt.legend()
                        plt.savefig(f"{folder_name}/loss_curve.png")
                        plt.close()
                        
                        # Plot accuracy curve
                        plt.figure()
                        plt.plot(range(epochs), accuracy_history, label='Accuracy')
                        plt.xlabel('Epochs')
                        plt.ylabel('Accuracy')
                        plt.title('Accuracy Curve')
                        plt.legend()
                        plt.savefig(f"{folder_name}/accuracy_curve.png")
                        plt.close()
                        
                        # Plot and save confusion matrix
                        cm = confusion_matrix(y_true, y_pred)
                        plt.figure(figsize=(8,6))
                        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
                        plt.xlabel('Predicted')
                        plt.ylabel('Actual')
                        plt.title('Confusion Matrix')
                        plt.savefig(f"{folder_name}/confusion_matrix.png")
                        plt.close()
                        
                        # Save metrics in csv
                        with open(f"{folder_name}/metrics.csv", "w") as f:
                            f.write(f"ActivationFunction,HiddenSize,LearningRate,BatchSize,NunberOfEpochs,Test Accuracy,Execution Time(in sec)/n{act_name},({hidden_size[0]} and {hidden_size[1]}),{lr},{batch_size},{epochs},{test_acc},{execution_time}")
                            

Epoch 1/50 - Loss: 33.8507, Accuracy: 0.102
Epoch 2/50 - Loss: 2.5134, Accuracy: 0.101
Epoch 3/50 - Loss: 2.5122, Accuracy: 0.099
Epoch 4/50 - Loss: 2.5173, Accuracy: 0.102
Epoch 5/50 - Loss: 2.5132, Accuracy: 0.103
Epoch 6/50 - Loss: 2.5137, Accuracy: 0.101
Epoch 7/50 - Loss: 2.5118, Accuracy: 0.100
Epoch 8/50 - Loss: 2.5165, Accuracy: 0.103
Epoch 9/50 - Loss: 2.5152, Accuracy: 0.101
Epoch 10/50 - Loss: 2.5150, Accuracy: 0.100
Epoch 11/50 - Loss: 2.5227, Accuracy: 0.099
Epoch 12/50 - Loss: 2.5211, Accuracy: 0.102
Epoch 13/50 - Loss: 2.5146, Accuracy: 0.100
Epoch 14/50 - Loss: 2.5173, Accuracy: 0.100
Epoch 15/50 - Loss: 2.5152, Accuracy: 0.102
Epoch 16/50 - Loss: 2.5185, Accuracy: 0.100
Epoch 17/50 - Loss: 2.5199, Accuracy: 0.102
Epoch 18/50 - Loss: 2.5210, Accuracy: 0.099
Epoch 19/50 - Loss: 2.5141, Accuracy: 0.101
Epoch 20/50 - Loss: 2.5152, Accuracy: 0.099
Epoch 21/50 - Loss: 2.5106, Accuracy: 0.100
Epoch 22/50 - Loss: 2.5171, Accuracy: 0.099
Epoch 23/50 - Loss: 2.5170, Accuracy: 0.

----
----
----

# Results of all the Variations

| ActivationFunction | HiddenSize    | LearningRate | BatchSize | NumberOfEpochs | TestAccuracy        | ExecutionTimeInSeconds | LossCurve                                                                   | AccuracyCurve                                                                       | ConfusionMatrix                                                                        |
| ------------------ | ------------- | ------------ | --------- | -------------- | ------------------- | ---------------------- | --------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------- |
| relu               | (100 and 100) | 0.001        | 10        | 50             | 0.963702380657196   | 1221.1444935798645     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/loss_curve.png" alt="Loss"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion"> |
| relu               | (100 and 100) | 0.01         | 10        | 50             | 0.7210999727249146  | 657.3895020484924      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/loss_curve.png" alt="Loss">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">  |
| relu               | (100 and 100) | 0.1          | 10        | 50             | 0.09819955378770828 | 603.5365436077118      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">   |
| relu               | (100 and 100) | 1            | 10        | 50             | 0.10099950432777405 | 458.0438141822815      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR1_B10_E50/loss_curve.png" alt="Loss">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR1_B10_E50/accuracy_curve.png" alt="Accuracy">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR1_B10_E50/confusion_matrix.png" alt="Confusion">     |
| relu               | (100 and 160) | 0.001        | 10        | 50             | 0.9660019278526306  | 1157.8503274917603     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.001_B10_E50/loss_curve.png" alt="Loss"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion"> |
| relu               | (100 and 160) | 0.01         | 10        | 50             | 0.8288005590438843  | 487.95741868019104     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.01_B10_E50/loss_curve.png" alt="Loss">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">  |
| relu               | (100 and 160) | 0.1          | 10        | 50             | 0.10089950263500214 | 419.38601183891296     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.1_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">   |
| relu               | (100 and 160) | 1            | 10        | 50             | 0.09739954024553299 | 406.1002399921417      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR1_B10_E50/loss_curve.png" alt="Loss">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR1_B10_E50/accuracy_curve.png" alt="Accuracy">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_160_LR1_B10_E50/confusion_matrix.png" alt="Confusion">     |
| relu               | (100 and 60)  | 0.001        | 10        | 50             | 0.9645021557807922  | 954.3139564990997      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.001_B10_E50/loss_curve.png" alt="Loss">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion">  |
| relu               | (100 and 60)  | 0.01         | 10        | 50             | 0.9279025197029114  | 647.4823021888733      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.01_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">   |
| relu               | (100 and 60)  | 0.1          | 10        | 50             | 0.1134994700551033  | 612.3821523189545      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.1_B10_E50/loss_curve.png" alt="Loss">    | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">    | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">    |
| relu               | (100 and 60)  | 1            | 10        | 50             | 0.10089950263500214 | 500.68239283561707     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR1_B10_E50/loss_curve.png" alt="Loss">      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR1_B10_E50/accuracy_curve.png" alt="Accuracy">      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_60_LR1_B10_E50/confusion_matrix.png" alt="Confusion">      |
| relu               | (160 and 100) | 0.001        | 10        | 50             | 0.9704017043113708  | 1067.8954167366028     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.001_B10_E50/loss_curve.png" alt="Loss"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion"> |
| relu               | (160 and 100) | 0.01         | 10        | 50             | 0.6314007043838501  | 489.5189564228058      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.01_B10_E50/loss_curve.png" alt="Loss">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">  |
| relu               | (160 and 100) | 0.1          | 10        | 50             | 0.1134994700551033  | 384.7974236011505      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.1_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">   |
| relu               | (160 and 100) | 1            | 10        | 50             | 0.08919956535100937 | 361.02485752105713     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR1_B10_E50/loss_curve.png" alt="Loss">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR1_B10_E50/accuracy_curve.png" alt="Accuracy">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H160_100_LR1_B10_E50/confusion_matrix.png" alt="Confusion">     |
| relu               | (60 and 60)   | 0.001        | 10        | 50             | 0.9583021998405457  | 1105.6882936954498     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.001_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion">   |
| relu               | (60 and 60)   | 0.01         | 10        | 50             | 0.8588016033172607  | 877.531261920929       | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.01_B10_E50/loss_curve.png" alt="Loss">    | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">    | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">    |
| relu               | (60 and 60)   | 0.1          | 10        | 50             | 0.09819955378770828 | 668.0333743095398      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.1_B10_E50/loss_curve.png" alt="Loss">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">     |
| relu               | (60 and 60)   | 1            | 10        | 50             | 0.10279951989650726 | 560.3649253845215      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR1_B10_E50/loss_curve.png" alt="Loss">       | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR1_B10_E50/accuracy_curve.png" alt="Accuracy">       | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H60_60_LR1_B10_E50/confusion_matrix.png" alt="Confusion">       |

# Results of all the Variations

| ActivationFunction | HiddenSize    | LearningRate | BatchSize | NumberOfEpochs | TestAccuracy        | ExecutionTimeInSeconds | LossCurve                                                                   | AccuracyCurve                                                                       | ConfusionMatrix                                                                        |
| ------------------ | ------------- | ------------ | --------- | -------------- | ------------------- | ---------------------- | --------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------- |
| relu               | (100 and 100) | 0.001        | 10        | 50             | 0.963702380657196   | 1221.1444935798645     | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/loss_curve.png" alt="Loss"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy"> | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion"> |
| relu               | (100 and 100) | 0.01         | 10        | 50             | 0.7210999727249146  | 657.3895020484924      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/loss_curve.png" alt="Loss">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy">  | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion">  |
| relu               | (100 and 100) | 0.1          | 10        | 50             | 0.09819955378770828 | 603.5365436077118      | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/loss_curve.png" alt="Loss">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy">   | <img src="https://github.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion">   |

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Results of all the Variations</title>
    
</head>
<body>
    <h1>Results of all the Variations</h1>
    <table>
        <tr>
            <th>Activation Function</th>
            <th>Hidden Size</th>
            <th>Learning Rate</th>
            <th>Batch Size</th>
            <th>Number of Epochs</th>
            <th>Test Accuracy</th>
            <th>Execution Time (Seconds)</th>
            <th>Loss Curve</th>
            <th>Accuracy Curve</th>
            <th>Confusion Matrix</th>
        </tr>
        <tr>
            <td>relu</td>
            <td>(100 and 100)</td>
            <td>0.001</td>
            <td>10</td>
            <td>50</td>
            <td>0.963702380657196</td>
            <td>1221.1444935798645</td>
            <td><img src="results/relu_H100_100_LR0.001_B10_E50/loss_curve.png" alt="Loss"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/accuracy_curve.png" alt="Accuracy"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.001_B10_E50/confusion_matrix.png" alt="Confusion"></td>
        </tr>
        <tr>
            <td>relu</td>
            <td>(100 and 100)</td>
            <td>0.01</td>
            <td>10</td>
            <td>50</td>
            <td>0.7210999727249146</td>
            <td>657.3895020484924</td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/loss_curve.png" alt="Loss"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/accuracy_curve.png" alt="Accuracy"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.01_B10_E50/confusion_matrix.png" alt="Confusion"></td>
        </tr>
        <tr>
            <td>relu</td>
            <td>(100 and 100)</td>
            <td>0.1</td>
            <td>10</td>
            <td>50</td>
            <td>0.09819955378770828</td>
            <td>603.5365436077118</td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/loss_curve.png" alt="Loss"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/accuracy_curve.png" alt="Accuracy"></td>
            <td><img src="https://raw.githubusercontent.com/Harshit-Soni78/23UADS4121-Harshit_Soni-NNLAB-2025/blob/main/Exp-4/results/relu_H100_100_LR0.1_B10_E50/confusion_matrix.png" alt="Confusion"></td>
        </tr>
    </table>
</body>
</html>
